In [ ]:
import h5py
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import numpy as np

from gwpy.timeseries import TimeSeries
from gwpy.frequencyseries import FrequencySeries
from ripple.waveforms.IMRPhenomD import gen_IMRPhenomD_hphc

## Load generated data

True, time-delayed O3a detector noise plus an injected synthetic signal.
Truth parameters are kept under `/truth` for cross-checks.

In [ ]:
arrays = {}
with h5py.File("test_bbh_v3.h5", "r") as f:
    f.visititems(
        lambda name, obj: arrays.update({name: obj[...]})
        if isinstance(obj, h5py.Dataset) else None
    )

for name, arr in arrays.items():
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}")

ifo = "H1"
color = {"H1": "#ee0000", "L1": "#4ba6ff"}[ifo]
fs = 4096
N = arrays[f"{ifo}/strain"].shape[0]


## Inputs in time and frequency space

In [ ]:
strain_ts = TimeSeries(arrays[f"{ifo}/strain"], t0=0, sample_rate=fs)
psd_fs = FrequencySeries(arrays[f"{ifo}/psd"], f0=0, df=fs / N)

strain_ts.plot(
    title=f"{ifo} strain (signal + O3a noise)", ylabel="Strain", color=color,
).show()

In [ ]:
white = strain_ts.whiten(asd=psd_fs ** 0.5).bandpass(30, 400)
plot = white.plot(color=color)
plot.gca().set_xlim(6.0, 8.0)
plot.show()

qtrans = white.q_transform(frange=(20, 500), outseg=(6.25, 8), whiten=False)
plot = qtrans.plot(figsize=(8, 5), vmin=0, vmax=25)
ax = plot.gca()
ax.set_yscale("log")
ax.set_ylabel("Frequency [Hz]")
plot.colorbar(label="Normalized energy")
plot.show()

## Matched filtering

Build `d_full` and `h_full` on the segment rFFT grid, then run the proper
matched filter against the stored PSD as a baseline.

In [ ]:
df = fs / N
freqs = jnp.fft.rfftfreq(N, d=1.0 / fs)
band = (freqs >= 20) & (freqs <= fs / 2)
f_ref = 20.0

strain = jnp.asarray(np.asarray(strain_ts))
psd_arr = jnp.asarray(np.asarray(psd_fs))

# Truth parameters in ripple's BBH order: [Mc, eta, chi1, chi2, D, tc, phic, iota].
q = float(arrays["truth/mass_ratio"])
theta_truth = jnp.array([
    float(arrays["truth/chirp_mass"]),
    q / (1 + q) ** 2,
    float(arrays["truth/chi1"]),
    float(arrays["truth/chi2"]),
    float(arrays["truth/distance"]),
    float(arrays["truth/tc"]),
    float(arrays["truth/phic"]),
    float(arrays["truth/inclination"]),
])

# Template on the rFFT grid, projected through this IFO's antenna factors.
fplus = float(arrays[f"antenna/{ifo}/fplus"])
fcross = float(arrays[f"antenna/{ifo}/fcross"])
hp, hc = gen_IMRPhenomD_hphc(freqs[band].astype(jnp.float64), theta_truth, f_ref)
h_band = (fplus * hp + fcross * hc).astype(jnp.complex128)
h_full = jnp.zeros_like(freqs, dtype=jnp.complex128).at[band].set(h_band)

# Tukey(alpha=0.01) on the strain matches the PSD's segment-edge treatment
# without eating the merger (only the first/last 1% of samples are tapered).
n_tukey = int(round(0.01 * N / 2))
ramp = 0.5 * (1 - jnp.cos(jnp.pi * (jnp.arange(n_tukey) + 0.5) / n_tukey))
window_td = jnp.concatenate([ramp, jnp.ones(N - 2 * n_tukey), ramp[::-1]])
d_full = jnp.fft.rfft(strain * window_td) / fs

psd_safe = jnp.where(band, psd_arr, jnp.inf)

In [ ]:
inner_hh = 4.0 * df * jnp.real(jnp.sum(jnp.abs(h_full) ** 2 / psd_safe))
truth_snr = float(arrays[f"truth/snr_{ifo}"])
print(f"sqrt<h|h> (stored PSD): {float(jnp.sqrt(inner_hh)):.3f}")
print(f"truth {ifo} SNR        : {truth_snr:.3f}")

## Matched filter


In [ ]:
# z(t) = 4 ∫ conj(d̃) h̃ e^{2πift} / S_n(f) df
# ρ(t) = |z(t)| / sqrt(<h|h>)
#
# Zero-pad the one-sided integrand back to a full length-N complex spectrum
# (negative freqs = 0) before the inverse FFT, so the 4 · fs factor matches
# the continuous-time matched-filter normalisation.
integrand = jnp.conj(d_full) * h_full / psd_safe
z_t = 4.0 * fs * jnp.fft.ifft(
    jnp.zeros(N, dtype=jnp.complex128).at[: integrand.shape[0]].set(integrand)
)
rho_t = jnp.abs(z_t) / jnp.sqrt(inner_hh)
peak_idx = int(jnp.argmax(rho_t))

print(f"<h|h>               : {float(inner_hh):.3f}")
print(f"Matched-filter peak : {float(rho_t[peak_idx]):.3f}  at t = {peak_idx / fs:.4f} s")
print(f"True {ifo} SNR         : {truth_snr:.3f}")
